# Delay Target Analysis
**OTP · DELAY DISTRIBUTION · CANCELLATIONS · DATA STRATEGY**

---


## Inhalt

- [Setup](#setup)
- [Target Definition](#target-definition)
  - [Delay Overview — Per Year](#delay-overview-per-year)
- [Temporal Trends](#temporal-trends)
- [Distribution](#distribution)
  - [Log Transform](#log-transform)
  - [Arrival vs Departure Delay](#arrival-vs-departure-delay)
  - [Delay Delta — Detail](#delay-delta-detail)
  - [Origin Stop Bias — Evidence and Filter Rule](#origin-stop-bias-evidence-and-filter-rule)
  - [Extreme Values](#extreme-values)
- [On-Time Performance (OTP)](#on-time-performance-otp)
  - [Root Cause — The Problem Is in the Schedule, Not Operations](#root-cause-the-problem-is-in-the-schedule-not-operations)
  - [OTP per Line — Temporal Trend](#otp-per-line-temporal-trend)
- [Cancellations](#cancellations)
  - [Cancellations by Line](#cancellations-by-line)
  - [Trip-Level Validation](#trip-level-validation)
  - [Line 12 — Construction Period](#line-12-construction-period)
  - [Monthly Delay](#monthly-delay)
  - [Delay per Line — Temporal Trend](#delay-per-line-temporal-trend)
  - [Background: VBZ Schedule Changes](#background-vbz-schedule-changes)
  - [Investigation: Nov/Dec 2025 Anomaly (resolved 2026-05-20)](#investigation-novdec-2025-anomaly-resolved-2026-05-20)
- [Key Findings](#key-findings)


> **Central Question Q4:** What does the delay target look like? — Distribution, OTP, arrival vs. departure delay.

Analyse der Zielvariable `arrival_delay` — Verteilung, OTP-Baseline, Vergleich mit Departure Delay und Delay Delta, Ausfälle.

## Setup

| Variable | Inhalt | Verwendung |
|---|---|---|
| `lf_all` | Rohdaten — alle Halte inkl. Ausfälle, Sonderlinien | Mengengerüst, Basisanalysen |
| `lf_delay` | `canceled==False` | Delay-Analysen über das gesamte Netz |
| `lf_clean` | `canceled=False` · `stop_sequence>1` · kein Linie E/L50/L51 · `departure_delay`/`delay_delta` maskiert auf NaN für 14. Nov–23. Dez 2025 · `is_anomal` Flag | **Standard** für Trend- & Modellanalysen |

**`is_anomal`:** Boolean-Flag — `True` = Zeile liegt im Anomaliewindow (departure_delay unplausibel, arrival_delay trotzdem gültig).

**Sampling:** Distributions-Plots (Histogramme, Boxplots) arbeiten intern auf einem **100k-Sample** (`seed=42`)  
— alle Aggregationen (Ø Delay, OTP, Monatstabellen) laufen über den **vollen Scan** des LazyFrame.

**Aufrufmuster:**
```python
an.plot_xxx(lf_all, cfg)           # nur roh
an.plot_xxx(lf_all, lf_clean, cfg) # Vergleich roh vs. bereinigt
show_df(an.table_xxx(lf_all))      # Tabelle — nimmt lf_all oder lf_clean
```

In [ ]:
import numpy as np
from zh_tram_flow.notebook import *
import zh_tram_flow.analytics.target as an

TRAIN, TEST, lf, lf_all, lf_delay, lf_clean = setup_analysis("03_analysis_1-target")

SAMPLE_SMALL = lf.collect().sample(n=100_000, seed=42)
SAMPLE_LARGE = lf.collect().sample(n=500_000, seed=42)

%load_ext autoreload
%autoreload 2

## Target Definition

**Primäres Ziel:** `arrival_delay` — Sekunden Verspätung bei der Ankunft an einer Haltestelle (negativ = zu früh).

Eine verspätete Abfahrt kann noch ausgeglichen werden — eine verspätete Ankunft nicht.   
Sie trifft Fahrgäste direkt: verpasste Anschlüsse, geplatzte Termine, Folgeverspätungen.

| Column | Rolle | Beschreibung |
|:---|:---|:---|
| `arrival_delay` | **Primäres Ziel** | Sekunden Verspätung bei Ankunft — was Fahrgäste erleben |
| `departure_delay` | Feature | Sekunden Verspätung bei Abfahrt — Startzustand für den nächsten Abschnitt |
| `delay_delta` | Abgeleitetes Feature | `departure_delay - arrival_delay` — positiv = Verspätung wächst am Halt, negativ = Verspätung wird abgebaut |

**Was wir nicht direkt sehen:** ob eine Verspätung über mehrere Halte vollständig aufgeholt wurde. `delay_delta` liefert das Signal pro Halt, aber keine Trip-Level-Sicht.

### Delay Overview — Per Year

Größenordnungen im Überblick: mittlere Verspätung pro Halt und Jahr, Min/Max. Alle drei Jahre (2023–2025) aus Train + Test kombiniert.

In [ ]:
section_header('Delay Overview per year')

an.plot_delay_overview_per_year(lf_all, lf_clean, cfg)

log('Table all')
show_df(an.table_delay_overview_per_year(lf_all))

log('Table clean')
show_df(an.table_delay_overview_per_year(lf_clean))

# delay_delta ist bereits im Feature-Set (berechnet in 02_preparation)
# is_recovering kann bei Bedarf abgeleitet werden: delay_delta < 0


**Beobachtung — Gesamter Jahresvergleich**

 Der Jahresvergleich zeigt einen strukturellen Aufwärtstrend über alle drei Metriken.   
 
 **Arrival Delay:** 2023: +54.8s → 2024: +58,3 s → 2025 (bereinigt, Jan–Okt): +55.2s — 2025 liegt leicht unter 2024, was auf eine Stabilisierung hindeutet, nicht auf kontinuierliche Verschlechterung.   
 
 **Delay Delta (bereinigt):** +4.3s → +4.9s → +5.0s — moderater Aufwärtstrend. Das rohe 2025-Delta von +7.7s (Jahrestabelle oben) ist durch den Nov/Dez-Artefakt aufgebläht und **nicht** als echter Trend zu interpretieren — bereinigt ist der Anstieg von 2023 auf 2025 nur +0.7s. → Vertiefen in `03_analysis_3-temporal`.

**Beobachtung — Bereinigter Jahresvergleich**

**Was hier anders ist als im Plot oben:**

Der erste Plot verwendet `lf_all` — also alle Einträge inklusive stornierter Fahrten und dem GTFS-Vorbereitungsartefakt in Nov/Dez 2025. Dieser bereinigte Plot filtert beides heraus, um ein ehrlicheres Bild des echten Betriebsgeschehens zu zeigen.

**Was die Trendlinien zeigen:**

Die gestrichelten Linien durch die Balken-Mittelpunkte machen die Richtung jeder Metrik über die drei Jahre sichtbar auf einen Blick. Steigt die Linie — wird es im Schnitt schlechter. Fällt sie — besser.

**Kernbefund:**

Der bereinigte `arrival_delay` zeigt, dass **2024 der schlechteste gemessene Jahrgang ist** und 2025 (Januar bis Oktober) bereits wieder leicht besser abschneidet. Das ist ein wichtiger Unterschied zum Rohdaten-Plot: Ohne Bereinigung sieht 2025 durch den GTFS-Artefakt in Nov/Dez aufgebläht aus.

Der `delay_delta` steigt moderat von 2023 bis 2025 an — das bedeutet, Trams akkumulieren im Schnitt pro Halt etwas mehr Verspätung als drei Jahre zuvor. Der Anstieg ist aber klein (wenige Zehntelsekunden pro Halt) und weit entfernt von einem Alarm-Signal.

**Was das für den Report bedeutet:**

> Das VBZ-Netz läuft strukturell stabil nahe seinem OTP-Ziel. 2025 zeigt eine leichte Erholung gegenüber 2024. Es gibt keinen dramatischen Aufwärtstrend — aber auch keinen Puffer für aussergewöhnliche Belastungen (Schnee, Grossevents, November-Peak). Das ist die seriöse und belegbare Aussage.

## Temporal Trends


In [ ]:
section_header('Monthly Delays')
an.plot_monthly_delay(lf_all, lf_clean, cfg)
log('Table all')
show_df(an.table_monthly_delay(lf_all))
log('Stats all')
show_df(an.table_delay_stats(lf_all))
log('Table clean')
show_df(an.table_monthly_delay(lf_clean))
log('Stats clean')
show_df(an.table_delay_stats(lf_clean))


**Beobachtung:** Der monatliche Verlauf zeigt ab **14. November 2025** einen abrupten Sprung in `delay_delta_mean` (von ~6s auf ~16s am 14. Nov, ~26s im Dezember). Dies entspricht keiner organischen Saisonschwankung — die Kurve bricht aus dem langjährigen Muster aus. Entscheidend: nur `departure_delay` steigt; `arrival_delay` bleibt stabil. Alle 15 Linien gleichzeitig betroffen. **Nov–Dez 2025 aus Trendanalysen ausschließen.** → Untersuchung und Erklärung im Hintergrund-Block weiter unten.

**Beobachtung:** Ohne den Fahrplanwechsel-Artefakt zeigt sich ein klares saisonales Muster: **Winter-Peak (Dez/Jan)** und ein kleinerer **Frühlings-Peak (März)** sowie **Sommer-Peak (Juni)** — unterbrochen von einem relativen Tal in den Sommermonaten (Juli–August), das aber trotzdem auf hohem Niveau bleibt. Die gestrichelten Trendlinien bestätigen einen **strukturellen Aufwärtstrend** über alle drei Metriken — kein Einmaleffekt. `dep_delay` steigt am stärksten. Alle drei Metriken steigen: das System wird insgesamt langsamer, nicht nur an einzelnen Punkten. → Saison-Feature (Monat, Winter/Sommer-Flag) und Jahr als Features in Modell aufnehmen.

## Distribution


Grundform aller drei Delay-Spalten: Minimum, Maximum, Mittelwert, Median, Streuung. Basis für alle weiteren Analysen.

Wie stark verändern die Bereinigungsschritte die Verteilung? `lf_all` (alles) vs. `lf_clean` (canceled raus · Nov/Dez 2025 raus · Linie E raus · Starthalte raus).

In [ ]:
section_header('Delays Distributions')
an.plot_delay_distribution_comparison(lf_all, lf_clean, cfg)
log('Table all')
show_df(an.table_delay_stats(lf_all))
log('Table clean')
show_df(an.table_delay_stats(lf_clean))

**Beobachtung — Gesamter Zeitraum**
Alle drei Verteilungen sind rechtsschief — wenige extreme Verspätungen ziehen den Mittelwert deutlich über den Median.

**Frühankünfte und Starthalte-Verzerrung:**
> Die auffälligen Frühankünfte bis −200s in der Verteilung stammen hauptsächlich von **Starthaltestellen (Terminus/Wendeschleifen)**. Trams starten dort mit eingebautem Puffer — was als negative Delay-Werte erscheint.
>
> **Auswirkung auf Durchschnittswerte:** Diese Frühankünfte ziehen den Netz-Durchschnitt nach unten und beschönigen die tatsächliche Verspätungsperformance im laufenden Betrieb. Ein Delay-Durchschnitt von 56s wäre ohne Starthalte-Effekt **höher**.
>
> Bei `delay_delta` gilt zudem: `median(A−B) ≠ median(A) − median(B)`, daher erscheint die Diskrepanz grösser als erwartet.

→ Räumlich prüfen in `03_analysis_4-spatial`. Für Modellierung: `is_start_stop`-Filter oder `n_threshold` empfohlen.

**Beobachtung — Was die Bereinigung verändert**

Die sechs Histogramme zeigen dieselben drei Delay-Metriken — oben roh, unten bereinigt. Das macht den Effekt jedes Bereinigungsschritts direkt sichtbar.

**Arrival Delay:**
Der Mittelwert verschiebt sich nach der Bereinigung nach rechts (höher). Das klingt zunächst paradox — aber es macht Sinn: Die Starthalte haben negative Delay-Werte (Frühankünfte durch Fahrplanpuffer) die den Roh-Durchschnitt nach unten ziehen. Wenn wir sie herausnehmen, sehen wir die echte Verspätung im laufenden Betrieb.

**Delay Delta:**
Der −50s-Cluster (Bimodalität) verschwindet in `lf_clean` fast vollständig — das bestätigt, dass er ausschliesslich aus Starthaltestellen stammt. Die bereinigte Verteilung ist deutlich unimodaler und zeigt klarer, dass das Netz pro Halt im Schnitt Verspätung aufbaut.

**Was das für den Report bedeutet:**
> Wir berichten **beide Werte** — roh und bereinigt — mit expliziter Erklärung warum sie sich unterscheiden. Das ist transparent und methodisch sauber: Der rohe Wert zeigt was im Datensatz steht, der bereinigte Wert zeigt die echte Systemperformance.

In [ ]:
# Quantitativer Effekt der Bereinigungsschritte
import polars as pl

lf_nc = lf_all.filter(pl.col('canceled') == False)
n_nc  = lf_nc.select(pl.len()).collect().item()

steps = [
    ('stop_sequence == 1', lf_nc.filter(pl.col('stop_sequence') == 1)),
    ('Linie E',            lf_nc.filter(pl.col('line_name')    == 'E')),
    ('Nov/Dez 2025',       lf_nc.filter(
        (pl.col('operating_date').dt.year()  == 2025) &
        (pl.col('operating_date').dt.month() >= 11)
    )),
]

rows = []
for label, sub in steps:
    s = sub.select([
        pl.len().alias('n'),
        pl.col('arrival_delay').mean().alias('arr'),
        pl.col('delay_delta').mean().alias('delta'),
    ]).collect()
    rows.append({
        'Filter-Schritt': label,
        'N entfernt': f"{s['n'][0]:>10,}",
        'Anteil':     f"{s['n'][0]/n_nc:.2%}",
        'Ø arr_delay (entfernt)': f"{s['arr'][0]:+.1f}s",
        'Ø delay_delta (entfernt)': f"{s['delta'][0]:+.1f}s",
    })

# Gesamtvergleich lf_all(nc) vs lf_clean
m_all   = lf_nc.select([pl.col('arrival_delay').mean().alias('arr'), pl.col('delay_delta').mean().alias('delta')]).collect()
m_clean = lf_clean.select([pl.col('arrival_delay').mean().alias('arr'), pl.col('delay_delta').mean().alias('delta')]).collect()

import pandas as pd
df = pd.DataFrame(rows)
log('Bereinigungsschritte — Mengen & Delay-Werte der entfernten Gruppen')
show_df(df.set_index('Filter-Schritt'))

summary = pd.DataFrame([{
    '': 'lf_all (non-canceled)',
    'N':              f"{n_nc:,}",
    'Ø arrival_delay': f"{m_all['arr'][0]:+.1f}s",
    'Ø delay_delta':   f"{m_all['delta'][0]:+.1f}s",
}, {
    '': 'lf_clean',
    'N':               f"{lf_clean.select(pl.len()).collect().item():,}",
    'Ø arrival_delay': f"{m_clean['arr'][0]:+.1f}s",
    'Ø delay_delta':   f"{m_clean['delta'][0]:+.1f}s",
}]).set_index('')
log('Gesamtvergleich lf_all vs lf_clean')
show_df(summary)

**Beobachtung — Bereinigung verändert Tendenzen nicht, aber die Verteilungsform**

Die Bereinigungsschritte entfernen **4.8% der Daten** (Starthalte) und **0.003%** (Linie E). Der Effekt auf die Mittelwerte ist minimal: `arrival_delay` verschiebt sich um **+1.1s**, `delay_delta` um **−0.8s**.

**Was sich trotzdem ändert:**
- Das **bimodale Cluster bei −50s** in `delay_delta` (Terminus-Artefakt) verschwindet → sauberere Verteilungsform
- **Linie E** (Ø 149s, aber nur 2.511 Zeilen) hat rechnerisch keinen Einfluss auf den Netzschnitt
- **Nov/Dez 2025** entfernt den Fahrplanwechsel-Artefakt — relevant für Trend, nicht für Mittelwert

> **Fazit:** `lf_clean` liefert eine sauberere Datenbasis für Modellierung. Die **grundlegenden Tendenzen** (Aufwärtstrend, Saisonalität, Linienunterschiede) **verändern sich durch die Bereinigung nicht** — sie werden nur klarer sichtbar.

### Log Transform

In [ ]:
an.plot_log_transform(lf_all, cfg)
an.plot_log_transform(lf_clean, cfg)

### Arrival vs Departure Delay

Alle drei Delay-Spalten nebeneinander als Boxplot — zeigt Lage, Streuung und Ausreißer auf einen Blick. Bauen Halte im Durchschnitt Verspätung auf oder ab?

In [ ]:
an.plot_arrival_vs_departure(lf_all, cfg)
an.plot_arrival_vs_departure(lf_clean, cfg)

**Beobachtung:** `departure_delay` liegt konsistent über `arrival_delay` — Halte kosten Zeit. `delay_delta` ist zentriert nahe 0 mit breiter Streuung: die meisten Halte sind annähernd neutral, aber extreme Werte in beide Richtungen sind vorhanden. Die starke linke Flanke des Delta (starke Recovery) deutet auf wenige Halte mit großem Zeitgewinn hin — wahrscheinlich Endhalte oder Expresssegmente wo Trams Puffer aufholen.

### Delay Delta — Detail

Separate Betrachtung der `delay_delta` Verteilung im engen Bereich (±100s). Wie ist die Form — symmetrisch, bimodal, stark schief?

In [ ]:
an.plot_delay_delta_detail(lf_all, cfg)
an.plot_delay_delta_detail(lf_clean, cfg)

**Beobachtung:** Die Verteilung ist **bimodal** — zwei erkennbare Häufungspunkte:

**Häufungspunkt 1 — nahe 0s:** Neutrale Halte. Das Tram fährt weiter ohne nennenswerte Änderung zur Planzeit.

**Häufungspunkt 2 — um −50s:** Das sind die **Starthaltestellen (Terminus/Wendeschleife)**.

> **Warum sind Starthaltestellen ein Problem für unsere Analyse?**
> Ein Tram, das am Startpunkt einer Linie abfährt, hat per Definition noch keine Verspätung angesammelt. Weil Fahrpläne dort Pufferzeit einbauen, starten Trams oft etwas früher als geplant — was als „negativer delay_delta" erscheint. Diese Frühankünfte und Frühabfahrten an Starthaltestellen:
> - **Verzerren den Netz-Durchschnitt nach unten** (machen das System scheinbar besser als es ist)
> - **Sind kein echtes Performance-Signal** — sie messen keinen Betriebsfortschritt, sondern Fahrplan-Puffer
>
> Für eine saubere Analyse der Verspätungsakkumulation **sollten Starthaltestellen herausgefiltert werden** — sie verfälschen Durchschnitte und Mediane. Ohne diesen Cluster ist der mittlere Verspätungsaufbau pro Halt noch deutlich höher als +5s.

→ Räumlich prüfen in `03_analysis_4-spatial`: welche Haltestellen haben systematisch delta < −30s

### Origin Stop Bias — Evidence and Filter Rule


Erste Haltestelle jeder Fahrt (`stop_sequence == 1`) vs. alle weiteren Haltestellen — zeigt ob und wie stark Starthalte die Delay-Verteilung verzerren.

> **Warum dieser Vergleich?** Die bimodale `delay_delta`-Verteilung oben hat einen auffälligen Cluster bei −50s. Wenn dieser Cluster ausschliesslich aus Starthaltestellen stammt, sind die Durchschnittswerte des gesamten Netzes systematisch zu optimistisch — weil jede Fahrt mit einem künstlichen Puffer-Bonus beginnt.

In [ ]:
an.plot_start_stop_analysis(lf_all, cfg, ylim_density=(0, 0.03))

**Beobachtung — Starthalte-Verzerrung: Der Beweis**

**Was wir sehen:**

Das dritte Panel ist der entscheidende Beweis: Bei Starthaltestellen (`stop_sequence == 1`) sind deutlich mehr als 50% aller `delay_delta`-Werte negativ — bei normalen Haltestellen liegt dieser Anteil klar darunter. Das erste Panel zeigt den gleichen Befund als Histogramm: Die Verteilung der Starthalte ist deutlich nach links verschoben (Richtung negative Werte), die der normalen Halte ist symmetrischer um 0s.

**Was das bedeutet — einfach erklärt:**

Stell dir eine Tramlinie vor. Am Startpunkt wartet das Tram auf seine planmässige Abfahrtszeit. Der Fahrplan hat dort extra Pufferzeit eingeplant — das Tram kommt also oft etwas früher an als der Fahrplan verlangt. Das erscheint in den Daten als negativer Wert (z.B. −50s "zu früh"). Das ist kein Fehler im Betrieb — das ist eingebauter Fahrplan-Puffer.

Das Problem: Dieser negative Wert fliesst in unseren Netz-Durchschnitt ein und macht das System scheinbar pünktlicher als es im laufenden Betrieb tatsächlich ist. Jede Fahrt "kauft" sich am Start einen negativen Bonus, der über die gesamte Strecke abbezahlt wird.

**Die Konsequenz für unsere Analyse:**

> Wenn wir fragen *"Wie pünktlich ist das VBZ-Netz?"*, dann sollten wir die Starthaltestellen herausnehmen. Denn dort wird Systemleistung nicht gemessen — dort wird Fahrplan-Puffer aufgebraucht.
> Die echte Frage ist: Wie entwickelt sich der Delay vom zweiten Halt an?

**Filterregel für Modellierung und Reportmetriken:**

```
stop_sequence > 1   →  echter Betriebsfortschritt
stop_sequence == 1  →  Fahrplan-Puffer / Start-Logik → aus Delay-Baseline herauslassen
```

Der bereinigte Netz-Durchschnitt (ohne Starthalte) liegt entsprechend **höher** als die bisher genannten ~56s — das zeigt das System wie es im laufenden Betrieb tatsächlich performt.

**Endhalte bleiben drin:**

Die letzte Haltestelle einer Fahrt (`stop_sequence == max`) messen, wie viel Verspätung über die gesamte Strecke aufgebaut wurde. Das ist ein wertvolles Signal — und kein Artefakt.

**Kaskaden-Frage (✅ gelöst → F-NET-07 → `prev_trip_delay` in LightGBM v2):**

**Ergebnis:** `prev_trip_delay` wurde in LightGBM v2 als Feature implementiert und ist Feature #2 nach `stop_name` (Gain). MAE sank von 45.7s auf 18,56 s — der Kaskadeneffekt ist der stärkste Einzelbeitrag im Modell.


### Extreme Values

Wie viele Halte liegen jenseits relevanter Schwellwerte? Gibt es echte Ausreißer oder ist die Verteilung kontinuierlich?

In [ ]:
section_header("Extreme Values")

total = lf.select(pl.len()).collect().item()
thresholds = [120, 300, 600, 1800]
rows = []
for t in thresholds:
    r = lf.select([
        (pl.col("arrival_delay")    >  t).sum().alias("arr_late"),
        (pl.col("arrival_delay")    < -t).sum().alias("arr_early"),
        (pl.col("departure_delay")  >  t).sum().alias("dep_late"),
    ]).collect()
    rows.append({
        "Threshold":  f"> {t}s  ({t//60}min)",
        "Arr Late":   f"{r['arr_late'][0]:>10,.0f}  ({r['arr_late'][0]/total:.2%})",
        "Arr Early":  f"{r['arr_early'][0]:>10,.0f}  ({r['arr_early'][0]/total:.2%})",
        "Dep Late":   f"{r['dep_late'][0]:>10,.0f}  ({r['dep_late'][0]/total:.2%})",
    })

show_df(pd.DataFrame(rows))

**Beobachtung:** Die extremsten Werte (+3000s bis +5000s) sind nicht zwingend Messfehler — bei großflächigen Störungen (Unwetter, Netzausfälle, Unfälle) können echte Kumulationsverspätungen dieser Größenordnung auftreten. Interessant wäre ein späterer Abgleich mit externen Ereignis-Daten (Wetterdaten, Störungsmeldungen) in `03_analysis_3-temporal`: Fallen die Extremwert-Häufungen zeitlich mit dokumentierten Ereignissen zusammen? Starke Frühankünfte (−200s+) konzentrieren sich vermutlich auf Terminushalte mit Pufferzeit.

## On-Time Performance (OTP)

Anteil der Halte innerhalb ±120 Sekunden Planzeit — der offizielle KPI-Schwellwert der VBZ.

> **Woher kommt der 120-Sekunden-Wert?**
> Die VBZ verwendet ±120s (= ±2 Minuten) als Toleranzgrenze im eigenen Qualitätsbericht sowie im VDPW-Standard (Verband Deutscher Verkehrsunternehmen). Abweichungen unter 2 Minuten sind für Fahrgäste an der Haltestelle praktisch nicht spürbar — daher gilt alles ab +120s als „nicht pünktlich". Der Schwellwert ist im öffentlichen Nahverkehr Deutschland/Schweiz/Österreich branchenweit etabliert.
> **Quellen:** VBZ Qualitätsbericht 2023/2024 · VDPW-Standard Pünktlichkeitsmessung

Für `delay_delta`: Anteil der Halte, an denen Verspätung abgebaut, neutral oder aufgebaut wird.

In [ ]:
section_header('On-Time Performance')
log('Plot all')
an.plot_otp(lf_all, cfg)
log('Tabel all')
show_df(an.table_otp(lf_all))
log('Plot clean')
an.plot_otp(lf_clean, cfg)
log('Table clean')
show_df(an.table_otp(lf_clean))

**Beobachtung:** **87 Arrival-OTP** (Schwellwert: ±120s, VBZ-Standard) — ein solider Wert für ein urbanes Tramnetz. Zürich liegt damit im europäischen Spitzenfeld. Die Nicht-Pünktlichen sind fast ausschließlich verspätet (12.9%) — kaum zu früh (0.1%). Das System hat eine klare Bias Richtung Verspätung, was auf systemischen Puffermangel hinweist.

> **Einordnung:** 87% OTP bedeutet, dass das VBZ-Netz seinen eigenen Standard in ca. 87 von 100 Halten einhält. Kein „schlechtes" Netz — aber auch kein Spielraum für strukturellen Mehrbedarf ohne Fahrplananpassung.

Bei `delay_delta`: **71.2%** der Halte bauen Verspätung auf (Growing), nur 27.2% zeigen Recovery — das Netz hat systemisch zu wenig Puffer eingebaut. Kaskadenwirkungen (eine verspätete Fahrt verzögert die nächste) sind mit `trip_id` analysierbar (→ F-NET-07).

### Root Cause — The Problem Is in the Schedule, Not Operations

> **"Das Verspätungsproblem ist nicht operativ — es ist im Fahrplan eingebaut."**

Zwei Befunde, die sich gegenseitig erklären:

* **F-TARGET-03:** 71.5% aller Halte akkumulieren Delay (`delay_delta > 0`)
* **F-SPAT-08:** 71.3% aller Halte haben keine Verweilzeit (`dwell_time = 0s`)

Das ist kein Zufall — es ist die gleiche Zahl, weil es dasselbe Problem ist.  
Ohne Pufferzeit kann eine Tram verlorene Sekunden nicht aufholen. Das System akkumuliert zwangsläufig.

Das bedeutet: Wetter, Events und Tageszeit entscheiden *wann* es schlimmer wird —  
aber die strukturelle Schwäche entscheidet, *dass* es passiert.

### OTP per Line — Temporal Trend

Wie unterscheiden sich die Linien in ihrer Pünktlichkeit — und verändert sich die Spreizung über die Zeit? Linien 9, 10, 12, 17 hervorgehoben; alle anderen als Hintergrund-Layer. `canceled = True` ausgeschlossen.

In [ ]:
section_header('On-Time Performance per Line')

log('Plot all')
an.plot_otp_per_line(lf_all, cfg)
log('Tabel all')
show_df(an.table_otp_per_line(lf_all))

log('Plot clean')
an.plot_otp_per_line(lf_clean, cfg, ylim_otp=(70, 110)) 
log('Tabel clean')
show_df(an.table_otp_per_line(lf_clean))

**Beobachtung:** Die Spreizung zwischen den Linien ist erheblich.

**Linie E — Ausreisser und Entscheidung:**

| | Drin lassen | Herausnehmen |
|:---|:---|:---|
| **Pro** | Vollständiges Bild des Netzes | Präsentation klarer, Modell-Baseline nicht verzerrt |
| **Contra** | Verzerrt alle Durchschnitte stark (128–130s vs. ~56s Netzschnitt) | Versteckt einen echten Betriebsaspekt |

**Was ist Linie E?** Eine Entlastungs-/Verstärkerlinie, die nur bei Bedarf (Grossevents, Stosszeiten) eingesetzt wird. Sie ist planmässig im GTFS modelliert, weicht aber im Betrieb strukturell von allen Regellinien ab — weil sie keine festen Fahrzeiten einhalten kann (sie reagiert auf aktuelle Auslastung). OTP 56.2%, Ø Delay 128–130s.

**Entscheid: Linie E wird aus der Hauptanalyse und der Modellierung ausgeschlossen.** Begründung: strukturell nicht vergleichbar mit Regellinien. Im Report wird der Ausschluss explizit dokumentiert. Das ist methodisch sauber — nicht Datenverfälschung.

**Ohne Linie E** liegt das schlechteste reguläre Tram bei **Linie 11 (82.0% OTP)**, gefolgt von Linie 15 (84.7%) und Linie 8 (84.9%). Linie 12 zeigt während der Baustellenphase einen starken Einbruch — aber mit normaler OTP ausserhalb der Baustelle.

## Cancellations

`canceled = True` ist der Extremfall — faktisch unendliche Verspätung. Wie viele Ausfälle gibt es insgesamt?

In [ ]:
section_header("Cancellations")

cancellations = (
    lf
    .group_by("canceled")
    .agg(pl.len().alias("count"))
    .with_columns((pl.col("count") / pl.col("count").sum()).alias("share"))
    .sort("canceled")
    .collect()
)
log(cancellations.to_pandas().to_string())

**Beobachtung:** **6.2% Ausfallrate** (3.87 Mio. von 62.1 Mio. Halt-Ereignissen im Trainings-Set).

> **Was bedeutet „Artefakt"? — Einfache Erklärung:**
> Stell dir vor, ein Tram fährt wegen einer Baustelle nur bis zur Hälfte der Strecke. Ist das ein „Ausfall"?
> - **Vor Juli 2024:** VBZ erfasste das als `canceled = True` — auch wenn das Tram teilweise fuhr (sogenannte Kurzwendungen oder Teilausfälle).
> - **Ab Juli 2024:** Nur noch echte Komplett-Ausfälle (Tram fährt gar nicht) bekommen `canceled = True`.
>
> Das Ergebnis: Die „Ausfallrate" sieht auf einen Schlag viel besser aus — nicht weil der Betrieb besser wurde, sondern weil die Definition enger wurde. Zahlen vor und nach Juli 2024 sind deshalb nicht direkt vergleichbar.

Der Grossteil der 6.2% fällt in die pre-Juli-2024-Periode und ist auf diese Datendefinitions-Änderung zurückzuführen (→ F-TARGET-05). Mit `trip_id` ist analysierbar ob Ausfälle einzelne Halte oder ganze Fahrten betreffen — das zeigt der nächste Abschnitt.

### Cancellations by Line


In [ ]:
section_header('Cancellations by Line')
an.plot_cancellations_by_line(lf_all, cfg)
show_df(an.table_cancellations_by_line(lf_all))

# → Tabelle bereits in Zelle oben via an.table_cancellations_by_line(lf_all)

**Beobachtung:** Linie 12 sticht mit Abstand heraus — die Ausfallrate liegt rund 20× über dem Durchschnitt aller anderen Linien. Zeitliche Eingrenzung (Jahresvergleich) zeigt, dass dies auf eine **Baustellen-Phase Januar 2023 – Juni 2024** zurückzuführen ist (Streckensperrung, Ersatzverkehr). Ab Juli 2024 normalisiert sich die Rate auf ~0.2%. Für Modellierung und alle linienübergreifenden Ausfallstatistiken sollte dieser Zeitraum entweder gefiltert oder als eigenes Feature (`linie_12_baustelle`) kodiert werden. → Für spätere Analyse: Backlog-Eintrag für detaillierte Zeitraumvalidierung.

### Trip-Level Validation


Ist `canceled` wirklich ein Trip-Level-Flag — d.h. wenn eine Fahrt ausfällt, sind **alle** Halte dieser Fahrt als `canceled = True` markiert?  
Oder gibt es "gemischte" Trips wo nur ein Teil der Halte canceled ist (→ das wären die Kurzwendungen der pre-Juli-2024-Ära)?

Gruppierung nach `trip_id` + `operating_date` — jeder Trip wird als `fully_canceled` / `fully_active` / `mixed` klassifiziert. Vergleich pre/post Juli 2024 zeigt ob sich das Muster mit der Datendefinitions-Änderung ändert.

In [ ]:
section_header("Canceled — Trip-Level Validierung")
summary = an.plot_trip_level_validation(
    PATHS["raw"] / "zh-tram-data-master.parquet", cfg
)
show_df(summary)

**Beobachtung:** Die Trip-Level-Analyse liefert den direkten Beweis für die Datendefinitions-Änderung:

> **Was sagen die Zahlen?**
> - **Pre-Juli 2024:** 6.001 `mixed` Trips (= Kurzwendungen — nur ein Teil der Halte ist canceled) + 112.790 `fully_canceled` Trips.
> - **Ab Juli 2024:** Die `mixed` Trips verschwinden vollständig — es gibt nur noch `fully_canceled` (18.250) oder `fully_active`.
>
> Warum ist das der Beweis? Bei echten Komplett-Ausfällen wäre ein Trip entweder ganz ausgefallen oder ganz gefahren. Dass es vor Juli 2024 Tausende „teils-ausgefallene" Trips gibt und danach null — das ist kein zufälliges Muster. Das zeigt, dass VBZ die Definitionsregel geändert hat: Kurzwendungen wurden früher als „teilweise canceled" gezählt, danach gar nicht mehr.

Das erklärt die netzweite simultane Normalisierung der Cancellation-Rate ab Juli 2024 besser als jede Baustellen-Theorie. → F-TARGET-05, F-TARGET-11

### Line 12 — Construction Period


In [ ]:
section_header('Cancellation Rate Over Time')

an.plot_cancellation_rate_over_time(lf_all, cfg)

section_header("Delay per Linie — Zeitlicher Verlauf")

log('Plot all')
an.plot_delay_per_line_timeline(lf_all, cfg, ylim_arr=(20,220), ylim_dep=(20,200), ylim_delta=(0,40))

log('Table all')
show_df(an.table_delay_per_line_summary(lf_all))

log('Plot clean')
an.plot_delay_per_line_timeline(lf_clean, cfg, ylim_arr=(20,120), ylim_dep=(20,120), ylim_delta=(0,15))

log('Table clean')
show_df(an.table_delay_per_line_summary(lf_clean))

**Beobachtung:** Linie 10 und Linie 12 zeigen synchrones Verhalten — aber mit allen Linien sichtbar wird der eigentliche Befund klar: **fast das gesamte Netz** hatte erhöhte Ausfallraten vor Juli 2024. Linie 9 (~10%), Linie 17 (~11%), Linie 7 (~6%) — alle normalisieren gleichzeitig im Juli 2024 auf ~0.3%, obwohl sie völlig unterschiedliche Strecken fahren. Das ist das stärkste Argument gegen eine infrastrukturelle Erklärung.

**Hypothese: Datendefinitions-Änderung Juli 2024.** Vor diesem Datum wurden vermutlich Kurzwendungen, Teilausfälle und Betriebsanpassungen als `canceled` geführt — ab Juli 2024 nur noch echte Vollausfälle. Das erklärt die netzweite simultane Normalisierung besser als jede Baustellen-Theorie. → **F-TARGET-11**

---

**Die Delay-Zeitachse als Gegenprobe:** Wenn die erhöhten Ausfallraten vor Juli 2024 auf ein echtes operatives Problem zurückzuführen wären — Baustelle blockiert das Netz, Kaskadenstörungen, strukturelle Beeinträchtigung — dann müssten die `arrival_delay`-Werte im selben Zeitraum ebenfalls erhöht oder volatiler sein. Die Monthly-Delay-Zeitachse zeigt genau das Gegenteil: ein **kontinuierlicher, gleichmässiger Aufwärtstrend** ohne Bruch, ohne Plateau, ohne erkennbare Erhöhung vor Juli 2024. Die Verspätungswerte von 2023 liegen sogar leicht *unter* denen von 2024.

Das ist der entscheidende Beweis: wäre die Baustelle operativ spürbar gewesen, hätten wir es in den Delays gesehen. Da wir es nicht sehen, war die erhöhte Ausfallrate kein reales Netzproblem — sondern ein **Reporting-Artefakt**. Zwei unabhängige Metriken (Cancellations vs. Delays) erzählen nur unter der Datendefinitions-Hypothese eine konsistente Geschichte.

---

**Revidierte Strategie — `canceled`-Flag pre/post Juli 2024:**

| Strategie | Beschreibung | Pro | Con |
|:---|:---|:---|:---|
| **A — Feature kodieren** | `is_pre_july_2024 = 1` für alle Linien vor Jul 2024 | Daten bleiben, Modell bekommt Kontext | Modell muss Effekt lernen |
| **B — Zeitraum filtern** | Pre-Jul 2024 `canceled`-Records aus Training | Sauberste Baseline | Verliert ~18 Monate Daten |
| **C — canceled komplett ausschließen** | `canceled = True` Records aus Delay-Modell raus (haben keine sinnvollen Delay-Werte) | Sinnvoll — ausgefallene Fahrten haben kein `arrival_delay` | Cancellation-Modell separat behandeln |

**Entscheidung: Strategie A + C kombiniert.** `canceled = True` Records werden aus dem Delay-Modell ausgeschlossen (die haben keine sinnvollen Verspätungswerte). Für ein separates Cancellation-Modell wird `is_pre_july_2024` als Feature kodiert. → F-TARGET-05

### Monthly Delay

Wie entwickelt sich `delay_delta` über die Zeit? Gibt es Saisonalität, oder ist der Anstieg linear?

**Beobachtung:** Der monatliche Verlauf zeigt ab **November 2025** einen abrupten Sprung in `delay_delta_mean` (von ~5s auf ~17s im November, ~26s im Dezember). Dies entspricht keiner organischen Saisonschwankung — die Kurve bricht aus dem langjährigen Muster aus. Wahrscheinlichste Ursache: **Fahrplanwechsel Dezember 2025** (VBZ-Netzrestrukturierung j25→j26). Wenn neue Soll-Zeiten erst verzögert ins GTFS eingepflegt wurden, würden die Ist-Abweichungen künstlich aufgebläht erscheinen. **Nov–Dez 2025 aus Trendanalysen ausschließen.** → Bereinigte Ansicht folgt direkt unten.

**Beobachtung:** Ohne den Fahrplanwechsel-Artefakt zeigt sich ein klares saisonales Muster: **Winter-Peak (Dez/Jan)** und ein kleinerer **Frühlings-Peak (März)** sowie **Sommer-Peak (Juni)** — unterbrochen von einem relativen Tal in den Sommermonaten (Juli–August), das aber trotzdem auf hohem Niveau bleibt. Die gestrichelten Trendlinien bestätigen einen **strukturellen Aufwärtstrend** über alle drei Metriken — kein Einmaleffekt. `dep_delay` steigt am stärksten. Alle drei Metriken steigen: das System wird insgesamt langsamer, nicht nur an einzelnen Punkten. → Saison-Feature (Monat, Winter/Sommer-Flag) und Jahr als Features in Modell aufnehmen.

### Delay per Line — Temporal Trend

Wie entwickeln sich alle drei Delay-Metriken pro Linie über die Zeit? Zeigt welche Linien strukturell höhere Verspätungen haben und ob sich die Spreizung verändert. `canceled = True` ausgeschlossen.

**Beobachtung:** Die Streuung zwischen den Linien ist erheblich. **Dominanter Ausreisser: Linie E** mit Ø 128s Arrival Delay — rund 70s über dem Netzschnitt (~57s). Ohne Linie E liegen die regulären Linien zwischen ~47s (Linie 5) und ~68s (Linie 11). Das linienspezifische Muster ist **zeitlich stabil**: eine Linie die 2023 schlecht war, ist auch 2024 und 2025 schlecht. `delay_delta` zeigt die stärkste Spreizung: Linie 11 (+6.5s) und Linie 10 (+6.4s) akkumulieren Verspätung systematisch; Linie E (-0.3s) ist in dieser Metrik neutral. Der Nov/Dez 2025-Artefakt ist auch hier sichtbar — er betrifft alle Linien gleichzeitig, was ein netzweites Messsystem-Artefakt bestätigt (→ Untersuchung im Hintergrund-Block). → `line_name` und `month` sind die stärksten strukturellen Prädiktoren.

### Background: VBZ Schedule Changes

> **Was ist ein Fahrplanwechsel — einfach erklärt:**
> Zweimal im Jahr (Dezember + Juni) wechselt die VBZ den offiziellen Fahrplan — die sogenannten GTFS-Daten. Das sind die Solldaten: welche Linie fährt wann, wo, mit welchen Haltestellen.
> Im Dezember 2023 war das ein besonders grosser Wechsel: Linien 9, 11 und 13 bekamen neue Streckenführungen, neue Haltestellen und neue Fahrzeiten. Diesen Zeitraum nennen wir `j23 → j24`.
>
> **Warum ist das für unsere Analyse wichtig?**
> Vergleiche von Linie 11 vor und nach Dezember 2023 hinken — es ist faktisch eine andere Linie. Ein Delay-Anstieg von Linie 11 zwischen j23 und j24 könnte bedeuten: die neue Strecke ist langsamer, ODER der Fahrplan wurde nicht angepasst, ODER es ist ein Einlaufeffekt der neuen Haltestellen.
> Deshalb muss `gtfs_year` als Kontextvariable immer mitgedacht werden — es kodiert nicht nur Zeit, sondern auch Netzstruktur.

---

### Investigation: Nov/Dec 2025 Anomaly (resolved 2026-05-20)

**Ausgangsbefund aus den Daten:**

Der monatliche Verlauf zeigt ab exakt **14. November 2025** einen abrupten Sprung in `delay_delta_mean`:

| Datum | `delay_delta_mean` |
|:---|:---|
| 13. November 2025 | ~6s (normal) |
| 14. November 2025 | ~16s (**+10s über Nacht**) |
| 21. November 2025 | ~30s+ (Spitzenwert) |
| 23. Dezember 2025 | Rückkehr zur Normalverteilung |

**Was wir überprüft haben:**

| Frage | Befund |
|:---|:---|
| Betrifft es alle Linien? | **Ja** — alle 15 Linien gleichzeitig, kein linienspezifisches Muster |
| Steigt `arrival_delay` auch? | **Nein** — `arrival_delay` bleibt stabil. Nur `departure_delay` steigt. |
| Haben sich die Schedule-Werte geändert? | **Nein** — `arrival_schedule` und `departure_schedule` unverändert. Kein j26-Fahrplan im Datensatz. |
| Gibt es ein bekanntes Ereignis am 14. Nov? | **Kein publiziertes Ereignis gefunden.** Bahnhofquai-Baustelle begann erst 14. Dez 2025. |

**Was das bedeutet — die Logik:**

```
delay_delta = departure_delay − arrival_delay
           = (actual_dep − scheduled_dep) − (actual_arr − scheduled_arr)
           = actual_dwell_time − scheduled_dwell_time
```

Da `scheduled_dep` und `scheduled_arr` unverändert sind, muss `actual_dwell_time` gestiegen sein — Trams blieben ab Nov 14 länger an Haltestellen als geplant.

**Verworfene Hypothese:**

> *"j26-GTFS-Daten wurden schrittweise eingespeist, was die Soll-Zeiten veränderte."*

Datenbasiert widerlegt: Schedule-Werte zeigen keine Änderung ab November 2025. Die Anomalie liegt in den IST-Abfahrtzeiten, nicht in den SOLL-Zeiten.

**Wahrscheinlichste Erklärung (nicht abschliessend beweisbar ohne VBZ-interne Daten):**

Ab Mitte November 2025 begann VBZ die operative Vorbereitung auf den grössten Fahrplanwechsel in der Unternehmensgeschichte (14. Dez 2025: "Tramnetz Süd", 10 von 14 Linien geändert). Fahrereinweisungen, Probeumläufe und neue Turnusplanung können reale Verlängerungen der Standzeiten verursacht haben, während der SOLL-Referenzwert im Datensatz weiterhin j25 enthielt. Die Rückkehr zur Normalverteilung am **23. Dezember** (9 Tage nach j26) ist konsistent mit dieser Erklärung.

---

**Filterentscheidung — gewählte Strategie: Maskierung statt Ausschluss**

Statt Nov/Dez 2025 vollständig auszuschliessen, werden `departure_delay` und `delay_delta` im Anomaliewindow auf `NaN` gesetzt. `arrival_delay` — die Zielvariable — ist nicht betroffen und bleibt vollständig erhalten.

| Spalte | Behandlung |
|:---|:---|
| `departure_delay` | → `NaN` für 14. Nov – 23. Dez 2025 |
| `delay_delta` | → `NaN` für 14. Nov – 23. Dez 2025 |
| `arrival_delay` | ✅ unverändert — vom Phänomen nicht betroffen |
| `is_anomal` | ✅ Neues Boolean-Flag — `True` = im Anomaliewindow |

**Warum das funktioniert:** `departure_delay` und `delay_delta` sind **keine Modell-Features** (verifiziert aus `lgbm_v1_meta.json` — keines der 32 Features enthält diese Spalten). Ihre Maskierung kostet das Modell nichts.

**Was wir gewinnen — der quantitative Mehrwert:**

| | Ausschluss (alt) | Maskierung (neu) | Gewinn |
|:---|:---|:---|:---|
| Test-Set (2025) | ~25M Zeilen | ~30 M Zeilen | **+~5 M Zeilen (+~20%)** |
| November 2025 | vollständig fehlend | ✅ vorhanden | Ø arrival_delay ~58s |
| Dezember 2025 | vollständig fehlend | ✅ vorhanden | Fahrplanwechsel-Monat |

**Warum das für die Evaluierung kritisch ist:**
November war der schlechteste Monat im Netz (~58s Ø arrival_delay, weit über Jahresdurchschnitt). Ein Modell, das nicht auf dem schlechtesten Monat evaluiert wird, überschätzt die eigene Güte systematisch. +~5 M Testzeilen und vollständige November-Abdeckung sind kein Detailgewinn — das ist der Unterschied zwischen ehrlicher und beschönigter Modell-Evaluation.

**Umsetzung:** Zentral in `apply_lf_clean()` + `mask_departure_anomaly()` (→ `src/zh_tram_flow/data/cleaning.py`). Alle Notebooks via `setup_analysis()`. Kein hardcodierter Filter mehr.

## Key Findings



→ Vollständige Findings-Tabelle mit Impact und Handlungsempfehlungen in [`03_analysis_0-overview.ipynb`](03_analysis_0-overview.ipynb).

| ID | Finding | Status |
|:---|:---|:---|
| F-TARGET-01 | `arrival_delay` rechtsschiefe Verteilung — Median 42s vs. Mean 56.3s; Log-Transform empfohlen | done |
| F-TARGET-02 | `delay_delta` bimodal — zwei Modi: Recovery-Cluster ~−45s (netzweit, nicht terminus-spezifisch) und Akkumulations-Cluster ~+15s; Verteilung unverändert in `lf_clean` | done |
| F-TARGET-03 | **71.5%** `delay_delta > 0` — kein ausreichender Fahrplanpuffer | done |
| F-TARGET-04 | Scheduled Dwell-Time (`dep_schedule − arr_schedule`) im Datensatz vorhanden — Mehrheit der Werte ist 0s (kein Puffer eingeplant); Feature schwach, nur für Halte mit >0s relevant | done |
| F-TARGET-05 | `canceled`-Flag netzweit erhöht Jan 2023 – Jun 2024 — Datendefinitions-Änderung beim Anbieter | done |
| F-TARGET-06 | Nov–Dez 2025 Fahrplanwechsel-Artefakt (j25→j26 GTFS) — aus Analysen ausgeschlossen (Strategie A) | done |
| F-TARGET-07 | Extremwerte bis ±3600s — wahrscheinlich echte Grossstörungen (Unwetter, Netzausfälle), kein Messfehler | done |
| F-TARGET-08 | `trip_id` und `stop_sequence` jetzt im Master-Datensatz — Kaskaden- und Trip-Level-Analyse möglich | done |
| F-TARGET-09 | Bereinigte Trendanalyse Jan–Okt 2025: delta +4.6s→+5.1s→+5.1s (moderater Aufwärtstrend); vollständiges 2025 mit Nov/Dez-Artefakt: +7.9s (irreführend) | done |
| F-TARGET-10 | `arrival_delay` 2025 (Jan–Okt, bereinigt): **+55.8s** — leicht unter 2024 (+59.4s) → Stabilisierung im Ankunfts-Delay | done |
| F-TARGET-11 | Netzweite synchrone Erhöhung der Ausfallrate aller Linien vor Jul 2024 — Trip-Level-Validierung bestätigt: 6.001 `mixed` Trips pre-Jul (Kurzwendungen), 0 mixed ab Jul. Datendefinitions-Änderung bewiesen | done |
| F-TARGET-12 | **Linie E** ist massiver Ausreisser: 55.7% OTP, 130s Ø Delay — als Entlastungslinie separat behandeln | done |
| F-TARGET-13 | Bereinigung (`lf_clean`) verschiebt Mittelwerte minimal (+1.1s arr, −0.8s delta) — Starthalte 4.8%, Linie E 0.003% der Daten. Tendenzen unverändert, Verteilungsform bereinigt (kein −50s-Cluster) | done |
